In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import lightgbm as lgb

RANDOM_STATE = 42

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/demand-forecasting-kernels-only/sample_submission.csv
/kaggle/input/competitions/demand-forecasting-kernels-only/train.csv
/kaggle/input/competitions/demand-forecasting-kernels-only/test.csv


In [2]:
def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    numerator = np.abs(y_pred - y_true)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0

    ratio = np.zeros_like(numerator)
    mask = denominator != 0
    ratio[mask] = numerator[mask] / denominator[mask]

    return 100.0 * np.mean(ratio)


In [3]:
def lgb_smape(y_pred, dataset):
    y_true = dataset.get_label()
    return "SMAPE", smape(y_true, y_pred), False

In [4]:

train = pd.read_csv("/kaggle/input/competitions/demand-forecasting-kernels-only/train.csv", parse_dates=["date"])
test = pd.read_csv("/kaggle/input/competitions/demand-forecasting-kernels-only/test.csv", parse_dates=["date"])

train["is_train"] = 1
test["is_train"] = 0
test["sales"] = np.nan

full = pd.concat([train, test], sort=False, ignore_index=True)
full = full.sort_values(["store", "item", "date"]).reset_index(drop=True)

In [5]:
def add_date_features(df):
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["dayofyear"] = df["date"].dt.dayofyear
    df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)
    df["quarter"] = df["date"].dt.quarter
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
    df["is_month_start"] = df["date"].dt.is_month_start.astype(int)
    df["is_month_end"] = df["date"].dt.is_month_end.astype(int)
    # cyclical encodings so the model knows Dec is "close to" Jan
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)
    return df

In [6]:
def add_lag_roll_features(df):
    g = df.groupby(["store", "item"])["sales"]
    lag_days = [90, 91, 98, 105, 112, 119, 126, 182, 364, 371, 728]
    for lag in lag_days:
        df[f"lag_{lag}"] = g.shift(lag)
    shifted_sales = g.shift(90)
    g_shifted = shifted_sales.groupby([df["store"], df["item"]])
    roll_windows = [7, 14, 30, 90, 180]
    for win in roll_windows:
        df[f"roll_mean_{win}"] = g_shifted.transform(lambda x: x.rolling(win, min_periods=1).mean())
        df[f"roll_std_{win}"] = g_shifted.transform(lambda x: x.rolling(win, min_periods=1).std())
    store_item_mean = df[df["is_train"] == 1].groupby(["store", "item"])["sales"].mean()
    df["store_item_mean"] = df.set_index(["store", "item"]).index.map(store_item_mean)
    store_month_mean = (
        df[df["is_train"] == 1].groupby(["store", "month"])["sales"].mean()
    )
    df["store_month_mean"] = df.set_index(["store", "month"]).index.map(store_month_mean)
    item_month_mean = df[df["is_train"] == 1].groupby(["item", "month"])["sales"].mean()
    df["item_month_mean"] = df.set_index(["item", "month"]).index.map(item_month_mean)
    return df

In [7]:
full = add_date_features(full)
full = add_lag_roll_features(full)

feature_cols = [
    "store", "item",
    "year", "month", "day", "dayofweek", "dayofyear", "weekofyear",
    "quarter", "is_weekend", "is_month_start", "is_month_end",
    "month_sin", "month_cos", "dow_sin", "dow_cos",
    "lag_90", "lag_91", "lag_98", "lag_105", "lag_112", "lag_119", "lag_126", "lag_182", "lag_364", "lag_371", "lag_728",
    "roll_mean_7", "roll_std_7",
    "roll_mean_14", "roll_std_14",
    "roll_mean_30", "roll_std_30",
    "roll_mean_90", "roll_std_90",
    "roll_mean_180", "roll_std_180",
    "store_item_mean", "store_month_mean", "item_month_mean",
]

train_full = full[full["is_train"] == 1].copy()
test_full = full[full["is_train"] == 0].copy()

In [8]:
cutoff = train_full["date"].max() - pd.Timedelta(days=90)

tr = train_full[train_full["date"] <= cutoff]
val = train_full[train_full["date"] > cutoff]

X_tr, y_tr = tr[feature_cols], tr["sales"]
X_val, y_val = val[feature_cols], val["sales"]

print(f"Train rows: {len(X_tr):,} | Validation rows: {len(X_val):,}")


Train rows: 868,000 | Validation rows: 45,000


In [9]:
import lightgbm as lgb
RANDOM_STATE = 42
lgb_train = lgb.Dataset(X_tr, label=y_tr, categorical_feature=["store", "item"])
lgb_val = lgb.Dataset(X_val, label=y_val, categorical_feature=["store", "item"], reference=lgb_train)

params = {
    "objective": "tweedie",       
    "tweedie_variance_power": 1.1,
    "metric": "None",             
    "learning_rate": 0.03,
    "num_leaves": 128,
    "min_data_in_leaf": 50,
    "feature_fraction": 0.85,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "lambda_l2": 0.1,
    "seed": RANDOM_STATE,
    "verbosity": -1,
}

In [10]:
model = lgb.train(
    params,
    lgb_train,
    num_boost_round=3000,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train", "valid"],
    feval=lgb_smape,
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=True),
        lgb.log_evaluation(period=200),
    ],
)


Training until validation scores don't improve for 100 rounds
[200]	train's SMAPE: 12.727	valid's SMAPE: 12.6228
[400]	train's SMAPE: 12.4824	valid's SMAPE: 12.3967
[600]	train's SMAPE: 12.3599	valid's SMAPE: 12.3328
[800]	train's SMAPE: 12.263	valid's SMAPE: 12.3144
[1000]	train's SMAPE: 12.1738	valid's SMAPE: 12.306
[1200]	train's SMAPE: 12.0873	valid's SMAPE: 12.3021
[1400]	train's SMAPE: 12.0058	valid's SMAPE: 12.3005
Early stopping, best iteration is:
[1312]	train's SMAPE: 12.0409	valid's SMAPE: 12.2995


In [11]:
val_pred = model.predict(X_val, num_iteration=model.best_iteration)
val_pred = np.clip(val_pred, 0, None) 

val_smape = smape(y_val.values, val_pred)
print(f"\nValidation SMAPE: {val_smape:.4f}%")


Validation SMAPE: 12.2995%


In [12]:
X_all, y_all = train_full[feature_cols], train_full["sales"]
lgb_all = lgb.Dataset(X_all, label=y_all, categorical_feature=["store", "item"])

final_model = lgb.train(
    params,
    lgb_all,
    num_boost_round=model.best_iteration,
)

test_pred = final_model.predict(test_full[feature_cols])
test_pred = np.clip(test_pred, 0, None)
test_pred = np.round(test_pred).astype(int)

submission = pd.DataFrame({"id": test_full["id"].astype(int), "sales": test_pred})
submission = submission.sort_values("id").reset_index(drop=True)

In [13]:
submission.to_csv("submission.csv", index=False)